In [13]:
import os
import oracledb
from dotenv import load_dotenv
import csv
from datetime import datetime

load_dotenv()

connection = oracledb.connect(
    user=os.getenv("ORACLE_USER"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=oracledb.makedsn("localhost", 1522, service_name="stu"),
)
cursor = connection.cursor()
print("Connected to Oracle")

Connected to Oracle


# setup tables

In [ ]:
ANES_2020 = "anes_2020"
ANES_2024 = "anes_2024"
TABLES = [ANES_2020, ANES_2024]

def setup_tables():
    for table in TABLES:
        try:
            cursor.execute(f"DROP TABLE {table}")
            print(f"Dropped {table}")
        except oracledb.DatabaseError as e:
            print(f"{table} did not exist: {e}")
    cursor.execute("PURGE RECYCLEBIN")
    print("Recycle bin purged")

    cursor.execute(
        f"""
        CREATE TABLE {ANES_2020} (
            id_2020 NUMBER(10) PRIMARY KEY,
            lib_con_scale VARCHAR2(30),
            democrat_thermometer NUMBER(5,1),
            republican_thermometer NUMBER(5,1),
            youtube_use VARCHAR2(3),
            election_year NUMBER(4) NOT NULL
        )
    """
    )
    print(f"Created {ANES_2020}")

    cursor.execute(
        f"""
        CREATE TABLE {ANES_2024} (
            id_2024 NUMBER(10) PRIMARY KEY,
            id_2020 NUMBER(10),
            lib_con_scale VARCHAR2(30),
            democrat_thermometer NUMBER(5,1),
            republican_thermometer NUMBER(5,1),
            youtube_use VARCHAR2(3),
            election_year NUMBER(4) NOT NULL
        )
    """
    )
    print(f"Created {ANES_2024}")

# insert from csv to sql

In [10]:
def clean_float(val):
    try:
        return float(val) if val and str(val).strip() not in ("", "nan") else None
    except:
        return None


def clean_int(val):
    try:
        return int(float(val)) if val and str(val).strip() not in ("", "nan") else None
    except:
        return None


def clean_str(val, max_len=None):
    if not val or str(val).strip() in ("", "nan"):
        return None
    s = str(val).strip()
    return s[:max_len] if max_len else s


def insert_anes_2020():
    rows = []
    with open(
        "../../cleaned_data/anes/anes_2020_clean_sample.csv",
        newline="",
        encoding="utf-8",
    ) as f:
        for r in csv.DictReader(f):
            rows.append(
                (
                    int(r["id_2020"]),
                    clean_str(r["lib_con_scale"], 30),
                    clean_float(r["democrat_thermometer"]),
                    clean_float(r["republican_thermometer"]),
                    clean_str(r["youtube_use"], 3),
                    int(r["election_year"]),
                )
            )
    cursor.executemany(
        """
        INSERT INTO anes_2020
        (id_2020, lib_con_scale, democrat_thermometer,
         republican_thermometer, youtube_use, election_year)
        VALUES (:1,:2,:3,:4,:5,:6)
    """,
        rows,
    )
    print(f"Inserted {len(rows)} rows into anes_2020")


def insert_anes_2024():
    rows = []
    with open(
        "../../cleaned_data/anes/anes_2024_clean_sample.csv",
        newline="",
        encoding="utf-8",
    ) as f:
        for r in csv.DictReader(f):
            rows.append(
                (
                    int(r["id_2024"]),
                    clean_int(r["id_2020"]),
                    clean_str(r["lib_con_scale"], 30),
                    clean_float(r["democrat_thermometer"]),
                    clean_float(r["republican_thermometer"]),
                    clean_str(r["youtube_use"], 3),
                    int(r["election_year"]),
                )
            )
    cursor.executemany(
        """
        INSERT INTO anes_2024
        (id_2024, id_2020, lib_con_scale, democrat_thermometer,
         republican_thermometer, youtube_use, election_year)
        VALUES (:1,:2,:3,:4,:5,:6,:7)
    """,
        rows,
    )
    print(f"Inserted {len(rows)} rows into anes_2024")

In [ ]:
setup_tables()

insert_anes_2020()
insert_anes_2024()

connection.commit()

anes_2020 did not exist: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/
anes_2024 did not exist: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/
Recycle bin purged
Created anes_2020
Created anes_2024
Inserted 500 rows into anes_2020
Inserted 500 rows into anes_2024


# look at data

In [14]:
import pandas as pd

query = f"""
SELECT *
FROM {ANES_2020} c
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,ID_2020,LIB_CON_SCALE,DEMOCRAT_THERMOMETER,REPUBLICAN_THERMOMETER,YOUTUBE_USE,ELECTION_YEAR
0,201438,Conservative,0.0,85.0,Yes,2020
1,529105,Moderate,100.0,0.0,Yes,2020
2,452663,DK,0.0,100.0,No,2020
3,207245,Slightly liberal,70.0,0.0,Yes,2020
4,356789,Liberal,90.0,0.0,Yes,2020
...,...,...,...,...,...,...
495,228853,Conservative,0.0,100.0,Yes,2020
496,224868,DK,100.0,0.0,Yes,2020
497,203410,Liberal,85.0,0.0,Yes,2020
498,211329,Extremely Liberal,0.0,0.0,No,2020


In [15]:
import pandas as pd

query = f"""
SELECT *
FROM {ANES_2024} c
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,ID_2024,ID_2020,LIB_CON_SCALE,DEMOCRAT_THERMOMETER,REPUBLICAN_THERMOMETER,YOUTUBE_USE,ELECTION_YEAR
0,141404,None,Extremely conservative,0.0,100.0,Yes,2024
1,141777,None,Slightly conservative,60.0,0.0,Yes,2024
2,365029,None,Extremely Liberal,NaN,NaN,No,2024
3,237087,None,DK,NaN,NaN,None,2024
4,352166,None,Extremely Liberal,85.0,0.0,No,2024
...,...,...,...,...,...,...,...
495,140797,None,Slightly liberal,100.0,0.0,No,2024
496,232015,None,Moderate,15.0,80.0,Yes,2024
497,220702,None,Extremely Liberal,90.0,0.0,Yes,2024
498,217234,None,Liberal,100.0,0.0,Yes,2024


In [17]:
cursor.close()
connection.close()

InterfaceError: DPY-1006: cursor is not open